In [1]:
import os
print(os.getcwd())

/home/yk/Coding/hrt-datathon/yehor


In [ ]:
ALPHA=40000
SENSITIVITY=1.0
CLIP=0.5


import os
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler

DATA_DIR = "../data"

# ── 1. Load training data ─────────────────────────────────────────────────────

seen_train    = pd.read_parquet(os.path.join(DATA_DIR, "bars_seen_train.parquet"), engine="fastparquet")
unseen_train  = pd.read_parquet(os.path.join(DATA_DIR, "bars_unseen_train.parquet"), engine="fastparquet")

# ── 2. Feature engineering ────────────────────────────────────────────────────

def build_features(bars: pd.DataFrame) -> pd.DataFrame:
    """Extract session-level features from seen OHLC bars."""

    def session_feats(grp):
        grp   = grp.sort_values("bar_ix")
        c     = grp["close"].values
        o     = grp["open"].values
        h     = grp["high"].values
        lo    = grp["low"].values
        n     = len(c)
        br    = np.diff(c) / c[:-1] if n > 1 else np.array([0.0])

        return pd.Series({
            "cum_return":   c[-1] / c[0] - 1,
            "last3_return": c[-1] / c[max(0, n - 4)] - 1,
            "slope":        np.polyfit(np.arange(n), c, 1)[0] / c.mean(),
            "realized_vol": np.std(br),
            "range_mean":   np.mean((h - lo) / c),
            "up_bar_frac":  np.mean(c > o),
            "wick_ratio":   np.mean((h - c) / (h - lo + 1e-8)),
        })

    return bars.groupby("session").apply(session_feats)

FEATURES = ["cum_return", "last3_return", "slope",
            "realized_vol", "range_mean", "up_bar_frac", "wick_ratio"]

# ── 3. Build training labels ──────────────────────────────────────────────────

mid_price = seen_train.groupby("session")["close"].last()
end_price = unseen_train.groupby("session")["close"].last()
y_train   = (end_price / mid_price - 1).rename("return")

X_train_raw = build_features(seen_train)[FEATURES]

# Align index (drop any sessions missing from either side)
idx = X_train_raw.index.intersection(y_train.index)
X_train_raw = X_train_raw.loc[idx]
y_train     = y_train.loc[idx]

# ── 4. Cross-validate to confirm we beat the baseline ────────────────────────

from sklearn.model_selection import KFold

def compute_sharpe(positions, returns):
    pnl = positions * returns
    return np.mean(pnl) / np.std(pnl) * 16

def make_positions(raw_preds, long_bias=1.0, sensitivity=0.3, clip=2.5):
    """
    Never go short (strong positive drift).
    Modulate position size around the long bias using model signal.
    position = long_bias + sensitivity * z_score(prediction)
    Clipped so minimum position is always > 0.
    """
    z = (raw_preds - raw_preds.mean()) / (raw_preds.std() + 1e-8)
    z = np.clip(z, -clip, clip)
    positions = long_bias + sensitivity * z
    positions = np.maximum(positions, 0.05)   # never fully flat
    return positions

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_train_raw)
y        = y_train.values

kf      = KFold(n_splits=5, shuffle=True, random_state=42)
cv_sharpes_model    = []
cv_sharpes_baseline = []

for train_idx, val_idx in kf.split(X_scaled):
    model = Ridge(alpha=ALPHA)
    model.fit(X_scaled[train_idx], y[train_idx])

    preds     = model.predict(X_scaled[val_idx])
    positions = make_positions(preds, sensitivity=SENSITIVITY, clip=CLIP)
    y_val     = y[val_idx]

    cv_sharpes_model.append(compute_sharpe(positions, y_val))
    cv_sharpes_baseline.append(compute_sharpe(np.ones(len(y_val)), y_val))

print(f"Baseline CV Sharpe : {np.mean(cv_sharpes_baseline):.4f} ± {np.std(cv_sharpes_baseline):.4f}")
print(f"Model    CV Sharpe : {np.mean(cv_sharpes_model):.4f} ± {np.std(cv_sharpes_model):.4f}")

# ── 5. Fit final model on all training data ───────────────────────────────────

final_model = Ridge(alpha=ALPHA)
final_scaler = StandardScaler()
X_final = final_scaler.fit_transform(X_train_raw)
final_model.fit(X_final, y)

print("\nFeature coefficients:")
for feat, coef in zip(FEATURES, final_model.coef_):
    print(f"  {feat:20s}  {coef:+.6f}")

# ── 6. Generate predictions for both test sets ────────────────────────────────

def predict_positions(bars: pd.DataFrame) -> pd.Series:
    feats    = build_features(bars)[FEATURES]
    X        = final_scaler.transform(feats)
    preds    = final_model.predict(X)
    positions = make_positions(preds)
    return pd.Series(positions, index=feats.index, name="target_position")

public_bars  = pd.read_parquet(os.path.join(DATA_DIR, "bars_seen_public_test.parquet"),  engine="fastparquet")
private_bars = pd.read_parquet(os.path.join(DATA_DIR, "bars_seen_private_test.parquet"), engine="fastparquet")

public_positions  = predict_positions(public_bars)
private_positions = predict_positions(private_bars)

# ── 7. Write submissions ──────────────────────────────────────────────────────

def write_submission(positions: pd.Series, path: str):
    sub = positions.reset_index()
    sub.columns = ["session", "target_position"]
    sub = sub.sort_values("session")
    sub.to_csv(path, index=False)
    print(f"Saved {len(sub)} rows → {path}")
    print(sub.describe())

write_submission(public_positions,  "submission_public.csv")
write_submission(private_positions, "submission_private.csv")

Baseline CV Sharpe : 2.7838 ± 0.5857
Model    CV Sharpe : 2.9824 ± 0.5270

Feature coefficients:
  cum_return            -0.000033
  last3_return          +0.000022
  slope                 -0.000032
  realized_vol          +0.000035
  range_mean            +0.000038
  up_bar_frac           -0.000028
  wick_ratio            +0.000033
Saved 10000 rows → submission_public.csv
           session  target_position
count  10000.00000     10000.000000
mean    5999.50000         0.999067
std     2886.89568         0.296346
min     1000.00000         0.250000
25%     3499.75000         0.790253
50%     5999.50000         0.983981
75%     8499.25000         1.200229
max    10999.00000         1.750000
Saved 10000 rows → submission_private.csv
           session  target_position
count  10000.00000     10000.000000
mean   15999.50000         0.999158
std     2886.89568         0.296497
min    11000.00000         0.250000
25%    13499.75000         0.794812
50%    15999.50000         0.985653
75%   